# Experiment 12 — four-model Beer Game study

**80 fictional testers / 20 chains per model, 24 decision weeks.**
Models: GPT-6 Luna, GPT-5.6 Luna, GPT-5.6 Sol and Claude Haiku 5.5.

**Current follow-up: 3 accepted GPT pilots are reused; 5 new calls remain (1 Claude pilot + 4 main calls).** One main API call per model produces all 20 chains. All four pilots must pass before main collection. The original plan had 8 calls; previous attempts and this follow-up share the same **$5 cap**, including technical retries.

**Recovery update:** If the saved run has the two failed GPT-6/Claude pilots, this notebook reuses the two accepted GPT-5.6 pilots and prepares **2 revised pilot calls + 4 main calls**. All original responses and charges remain preserved. Close and reopen this notebook to load the updated cells, then restart the kernel before running it.

**Latest update:** If GPT-6 passed that recovery but Claude failed inventory calculations, the notebook selects the bounded Claude follow-up: **reuse 3 accepted GPT pilots, run 1 revised Claude pilot, then 4 main calls if all pilots pass**. Claude receives medium effort and explicit physical bookkeeping. The existing $5 budget still covers every stage.

This compares **imagined no pressure versus strong pressure**. Both conditions use the same actual API settings. The model generates both conditions together, so results describe a scenario contrast; they do not establish binding time, independent students or statistical significance.

Run cells from top to bottom. Review is free by default. To collect data, change `RUN_API` below to `True`; the next section asks for your two API keys in hidden fields. Never paste keys into a code cell or chat.

[Experiment guide](../README.md) · [Hypotheses and protocol](../hypotheses/PROTOCOL.md) · [Latest report](../reports/REPORT.md)

## 1. Setup

Use the Python environment containing the project requirements. If imports fail, install them with `%pip install -r ../../../requirements.txt`, then restart the kernel and run again.

In [1]:
from pathlib import Path
import sys, os, json, getpass
import pandas as pd
from IPython.display import display, Markdown, HTML

here = Path.cwd().resolve()
candidates = [here, *here.parents, here / "Project" / "beer_game_experiment", here / "beer_game_experiment"]
ROOT = next((p for p in candidates if (p / "src/beergame_study/batched_pressure.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the Beer Game project folder.")
sys.path.insert(0, str(ROOT / "src"))
from beergame_study import batched_pressure as study
from beergame_study.batched_pressure_api import Runner, estimate
from beergame_study.batched_pressure_report import build_report
from beergame_study import batched_pressure_recovery as recovery
from beergame_study import batched_pressure_followup as followup

EXPERIMENT = ROOT / "experiments/12_pressure_framing_api_20261020"
print("Experiment folder:", EXPERIMENT)

Experiment folder: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/experiments/12_pressure_framing_api_20261020


## 2. Prepare and check the plan

`RUN_API = False` reads/prepares files without paid calls. Change it to `True` when ready.

`PHASE = "all"` finishes the pilots, checks their format/rules/physical state, and then runs the four main requests only if every pilot passes. Use `"pilot"` to stop after the pilot. The notebook detects the latest saved stage: the current Claude follow-up reuses three accepted GPT pilots and schedules one revised Claude pilot. Completed requests are reused on resume. The $5 cap includes every run and retry in this experiment folder.

In [2]:
RUN_API = False
PHASE = "all"  # "all", "pilot", or "main"

run_dir = study.prepare(EXPERIMENT, cap_usd=5.0)
plan = study.loadplan(run_dir)
old_status_path = run_dir / "api_status.json"
old_tasks = json.loads(old_status_path.read_text()).get("tasks", {}) if old_status_path.exists() else {}
failed_pilots = {name for name, row in old_tasks.items() if name.endswith("_pilot") and row.get("status") == "invalid"}
recovery_dir = recovery.prepare(run_dir) if failed_pilots == {"gpt-6-luna_pilot", "claude-haiku-5-5_pilot"} else None
followup_dir = None
if recovery_dir and (recovery_dir / "api_status.json").exists():
    recovery_tasks = json.loads((recovery_dir / "api_status.json").read_text())["tasks"]
    gpt_models = ("gpt-6-luna", "gpt-5.6-luna", "gpt-5.6-sol")
    if (all(recovery_tasks.get(m + "_pilot", {}).get("status") == "accepted" for m in gpt_models)
            and recovery_tasks.get("claude-haiku-5-5_pilot", {}).get("status") == "invalid"):
        followup_dir = followup.prepare(recovery_dir)
print("Frozen study:", plan["study_id"])
if followup_dir:
    print("Claude follow-up: reuse 3 accepted pilots; 1 revised Claude pilot + 4 main calls")
    print("Follow-up records:", followup_dir)
    print("Claude: medium adaptive effort and explicit stock/backlog/delivery bookkeeping.")
elif recovery_dir:
    print("Recovery: reuse 2 accepted pilots; 2 revised pilot calls + 4 main calls")
    print("Recovery records:", recovery_dir)
    print("Original failures remain saved. GPT-6: medium reasoning; Claude: larger output allowance.")
else:
    print("Planned: 4 pilot responses + 4 main responses")
print("Main: 80 testers × 4 models = 320 fictional role identities; 7,680 weekly orders")
costs = pd.DataFrame(followup.estimate(followup_dir) if followup_dir else recovery.estimate(recovery_dir) if recovery_dir else estimate(run_dir))
display(costs)
print("Conservative first-attempt reservation total: $" + f"{costs.first_attempt_reservation_ceiling_usd.sum():.4f}")
print("Hard total cap, including retries: $5.00")

Frozen study: pressure_api_c04ef16c05dd826c
Claude follow-up: reuse 3 accepted pilots; 1 revised Claude pilot + 4 main calls
Follow-up records: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/experiments/12_pressure_framing_api_20261020/data/pressure_api_c04ef16c05dd826c/followup/pressure_followup_aa5649bfb1e797ee
Claude: medium adaptive effort and explicit stock/backlog/delivery bookkeeping.
Main: 80 testers × 4 models = 320 fictional role identities; 7,680 weekly orders


,model,pilot_calls,main_calls,first_attempt_reservation_ceiling_usd,note
0,gpt-6-luna,0,1,0.020781,Five new fixed slots; three accepted pilots re...
1,gpt-5.6-luna,0,1,0.028119,Five new fixed slots; three accepted pilots re...
2,gpt-5.6-sol,0,1,0.496420,Five new fixed slots; three accepted pilots re...
3,claude-haiku-5-5,1,1,0.060323,Five new fixed slots; three accepted pilots re...


Conservative first-attempt reservation total: $0.6056
Hard total cap, including retries: $5.00


## 3. API keys and collection

**This is the API-key input cell.** With `RUN_API=True`, it requests the OpenAI key, then the Anthropic key. Paste each into its hidden input field and press Enter.

The progress bar counts accepted slots (including reused pilots), failures, blocked requests and requests left. A failed technical pilot blocks all main calls; its original response remains available for diagnosis. Null or opposite hypothesis results are accepted normally. Both providers' keys are needed to complete the four-model pilot.

For the first recovery, GPT-6 uses medium effort and a precise accounting appendix; Claude keeps low effort with more room for reasoning plus JSON. The Claude follow-up changes Claude to medium effort with explicit accounting. The two GPT-5.6 models retain their original settings. Each stage has a fixed generation allowance; the notebook does not silently repeat a newly invalid response.

In [3]:
if RUN_API:
    keys = {}
    runner = None
    try:
        keys["openai"] = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key (hidden): ").strip()
        keys["anthropic"] = os.environ.get("ANTHROPIC_API_KEY") or getpass.getpass("Anthropic API key (hidden): ").strip()
        if not all(keys.values()):
            raise ValueError("Both keys are required; no requests have been sent.")
        progress_display = display(HTML("Starting saved-progress check…"), display_id=True)
        def show_progress(p):
            text = (f"{p['done']}/{p['total']} finished · {p['remaining']} left · "
                    f"{p['accepted']} accepted · {p['failed']} failed · {p['skipped']} blocked · "
                    f"${p['charged_or_reserved_usd']:.4f} / $5.00")
            progress_display.update(HTML(f'<progress value="{p["done"]}" max="{p["total"]}" style="width:100%"></progress><p>{text}</p>'))
        runner_type = followup.Runner if followup_dir else recovery.Runner if recovery_dir else Runner
        runner = runner_type(followup_dir or recovery_dir or run_dir, keys=keys, callback=show_progress)
        execution = await runner.execute(PHASE)
        print("Collection records:", followup_dir or recovery_dir or (EXPERIMENT / "raw" / run_dir.name))
        published = build_report(run_dir, recovery=recovery_dir, followup=followup_dir)
        print("Report updated automatically:", published["report"])
    finally:
        keys.clear()
        if runner is not None:
            runner.keys.clear()
        del keys
else:
    print("Review mode: no API requests and no key input. Change RUN_API to True in section 2 to collect data.")

Review mode: no API requests and no key input. Change RUN_API to True in section 2 to collect data.


## 4. Findings and detailed results

Run this section even if collection stopped. Missing results remain missing. H1 is a pipeline-sensitivity proxy; H2 is the cost calculated by Python from the submitted orders. Negative H1 and positive H2 differences follow the predicted directions. The report never labels a trend as proof or reports independent-student p-values.

In [4]:
published = build_report(run_dir, recovery=recovery_dir, followup=followup_dir)
summary = published["summary"]
display(pd.DataFrame(summary["model_status"]))
display(pd.DataFrame(summary["hypotheses"])[["model", "hypothesis", "paired_blocks", "mean_strong_minus_no", "descriptive_direction"]])
print("Full report:", published["report"])
print("Excel results:", published["workbook"])
report_href = os.path.relpath(published["report"], EXPERIMENT / "notebooks")
workbook_href = os.path.relpath(published["workbook"], EXPERIMENT / "notebooks")
display(Markdown(f"[Open full report]({report_href}) · [Download Excel results]({workbook_href})"))
print("Conservative charges/reservations: $" + f"{published['charged_or_reserved_usd']:.4f}")
landing = (EXPERIMENT / "reports/REPORT.md").read_text(encoding="utf-8")
display(Markdown(landing.replace("](history/", "](../reports/history/")))

,model,accepted_pilot,expected_pilot,accepted_main,expected_main,complete,accepted_pilot_responses,accepted_main_responses,inference_available
0,gpt-6-luna,2,2,0,20,False,1,0,False
1,gpt-5.6-luna,2,2,0,20,False,1,0,False
2,gpt-5.6-sol,2,2,0,20,False,1,0,False
3,claude-haiku-5-5,0,2,0,20,False,0,0,False


,model,hypothesis,paired_blocks,mean_strong_minus_no,descriptive_direction
0,gpt-6-luna,H1,0,None,unavailable
1,gpt-6-luna,H2,0,None,unavailable
2,gpt-5.6-luna,H1,0,None,unavailable
3,gpt-5.6-luna,H2,0,None,unavailable
4,gpt-5.6-sol,H1,0,None,unavailable
5,gpt-5.6-sol,H2,0,None,unavailable
6,claude-haiku-5-5,H1,0,None,unavailable
7,claude-haiku-5-5,H2,0,None,unavailable


Full report: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/experiments/12_pressure_framing_api_20261020/reports/history/6dfb729f1751/REPORT.md
Excel results: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/experiments/12_pressure_framing_api_20261020/analysis/pressure_api_c04ef16c05dd826c__pressure_followup_aa5649bfb1e797ee/6dfb729f1751/RESULTS.xlsx


[Open full report](../reports/history/6dfb729f1751/REPORT.md) · [Download Excel results](../analysis/pressure_api_c04ef16c05dd826c__pressure_followup_aa5649bfb1e797ee/6dfb729f1751/RESULTS.xlsx)

Conservative charges/reservations: $0.0759


# Experiment 12: latest findings

1. **Collection:** 0/4 main responses and 3/4 pilot responses accepted.
2. **H1/H2 findings:** No main API results have been collected. Software checks do not provide hypothesis evidence.
3. **Interpretation:** This tests imagined pressure framing. It does not establish a binding deadline, original structural β, or effects on real students.
4. **Statistical independence:** All 20 chains and both conditions share one main response per model. No p-values or inferential confidence intervals are reported.
5. **Budget:** $0.0759 charged or conservatively reserved out of the authorized $5 total across this experiment, including pilot and retries.
6. **Claude follow-up:** Three accepted GPT pilots are reused. One revised Claude pilot is scheduled with medium effort and explicit physical bookkeeping; both earlier Claude failures remain preserved.

[Open the full report](../reports/history/6dfb729f1751/REPORT.md).


## 5. Inspect the exact prompts

Each file below contains the original API prompt and a separate JSON schema. All 20 main chains are in the single main prompt for that model. If recovery is active, its separate frozen request files include the revised prompts and settings; the original prompts stay unchanged. All stages share the same cumulative $5 cap.

In [5]:
display(pd.DataFrame([
    {"model": t["model"], "phase": t["phase"], "chains": len(t["chains"]),
     "prompt": str(EXPERIMENT / t["prompt_path"]), "schema": str(EXPERIMENT / t["schema_path"])}
    for t in plan["tasks"]
]))
if followup_dir or recovery_dir:
    print("Exact revised request payloads and amendment manifest:", followup_dir or recovery_dir)

,model,phase,chains,prompt,schema
0,gpt-6-luna,pilot,2,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...
1,gpt-5.6-luna,pilot,2,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...
2,gpt-5.6-sol,pilot,2,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...
3,claude-haiku-5-5,pilot,2,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...
4,gpt-6-luna,main,20,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...
5,gpt-5.6-luna,main,20,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...
6,gpt-5.6-sol,main,20,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...
7,claude-haiku-5-5,main,20,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...,/Users/gracezhang/Desktop/Y3S1/Project/beer_ga...


Exact revised request payloads and amendment manifest: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/experiments/12_pressure_framing_api_20261020/data/pressure_api_c04ef16c05dd826c/followup/pressure_followup_aa5649bfb1e797ee
